In [78]:
import os
import sys
import anndata as ad
import scipy
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import scipy.io as sio
import scanpy.external as sce
import matplotlib.pyplot as plt
import re
import gseapy as gp
import anndata as ad
import statistics
import tempfile
import sklearn
import cosg
import leidenalg
import celltypist
import muon as mu
from tqdm import tqdm
sc._settings.n_jobs= 24
sc.settings.verbosity = 1
# Adjust Scanpy figure defaults
sc.settings.set_figure_params(dpi=100, fontsize=10, dpi_save=400,
    facecolor = 'white', figsize=(8,8), format='png')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

In [79]:
def check_dict_duplicates(dict):
    seen = set()
    dup = any(item in seen or seen.add(item) for lst in dict.values() for item in lst)
    if dup==False:
        return '无重复'
    else:
        return '有重复'

In [80]:
obj_path = '/home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Level2_Refine_R8/'

In [81]:
finnal_path = '/home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Finnal/'

In [82]:
leiden_groups=['L4_leiden_TOTALVI_0.1','L4_leiden_TOTALVI_0.5', 'L4_leiden_TOTALVI_1', 'L4_leiden_TOTALVI_1.5', 'L4_leiden_TOTALVI_0.3','L4_leiden_TOTALVI_0.8']

In [83]:
def get_norm_annotation_data(celltype):
    adata = sc.read_h5ad(f"{obj_path}/{celltype}/{celltype}_count_scRNA.h5ad")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adt = sc.read_h5ad(f"{obj_path}/{celltype}/{celltype}_preprocess_scADT.h5ad")
    adata.obsm = adt.obsm
    
    # read Level1, tcr, bcr infor
    indices = pd.read_csv(f'{obj_path}/{celltype}/R8_indices_{celltype}.csv',index_col=0)
    # join leiden groups
    leiden_data = adt.obs.loc[:,leiden_groups]
    adata.obs = adata.obs.join(indices, how='left')
    adt.obs = adt.obs.join(indices, how='left')
    adata.obs = adata.obs.join(leiden_data, how='left')
    return adata,adt,leiden_data

# 1. 定 终 Th1 Cell Refine

In [84]:
celltype="Th1"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [85]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R4'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4,
           )

In [ ]:
adata.obs['Celltype_L4_L5_Refine_R3'].value_counts()

In [86]:
groupby = "L4_leiden_TOTALVI_0.5"

In [383]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [385]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='obs',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
sc.pl.umap(adt, color='HLA-DR',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',layer='dsb')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='CD183',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD185',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False,layer='dsb')
sc.pl.umap(adt, color='CD196',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False,layer='dsb')
sc.pl.umap(adt, color='CD161',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False,layer='dsb')
sc.pl.umap(adt, color='CD45RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD16',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False,layer='dsb')
sc.pl.umap(adt, color='IgM',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False,layer='dsb')
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='Celltype_L4_L5_Refine_R2',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='KLRB1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='GZMH',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='CXCR3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='CCR6',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.violin(adt,'CD27',size=0.1,groupby=groupby,layer='dsb')

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB',
                      'PDCD1','CTLA4','HAVCR2','LAG3','AHR','STAT1','GATA3','FOXP3','CD38','CCR5','TIGIT','ENTPD1','CD160',
                     'TCF7','TBX21','EOMES'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB',
                      'LTK','PTPN13','PDE4D','CCR6','RORC','NR1D1','CTSH','KIF5C','LGALS3','USP10','CMTM6','TOB1',
                     'TNFSF13B','CISH','AQP3','AUTS2','NSG1','S100A4'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1/Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'GZMH','IL18RAP','S1PR5','LYAR','NKG7','CST7','PRF1','TBX21','LINC01871','KLRG1','MYBL1','EOMES',
                     'EFHD2','DUSP2','SAMD3','CTSW','ID2','MATK','HOPX',],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','TBX21','IFNG',
                      'CMC1','CST7','FCRL3','CCL4','SLAMF7','EOMES','PDCD1','NKG7','CCR5','KLRK1','F2R','PLEK'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th22
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'AHR',
                      'CRIP1','LGALS1','LGALS3','S100A10','S100A4','PI16','LMNA','ANXA5','ANXA2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th2
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','AHR',
                      'PTGDR2','SNED1','NEFL','GATA3','FXYD7','C1orf162','GDPD5','IL4R','CAPG',
                     'LGALS1','TNFSF10','TNFRSF4','PPP1R9B','CSGALNACT1','NIBAN1','ERN1','SORL1','RUNX2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','LAG3','HAVCR2','BTBD9',
                      'CCL5','CTLA4','CD40LG'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th from BD
#TNFSF8=CD30L B3GAT1=CD57  BTLA= CD272  SLAMF5=CD84 HAVCR2=CD365
sc.pl.dotplot(adata, ['CXCR5','IL6R','TNFSF8','NRP1','IL21R','B3GAT1','BCL6','MAF','STAT3','ICOS','PDCD1','TIGIT','BTLA','CD200','SLAMF1','CD84',#Tfh
                      'IL4','IL17F','IL17A','IL21',#tfh分泌
                      'GATA3','SMAD1','STAT6','SPI1','IRF4',#Th9
                      'IL9','IL10','CCL17','CCL22','TGFB1',#th9分泌
                      'HAVCR2','CXCR4','CCR3','CCR4','CCR8','PTGDR2','GATA3','STAT5A','STAT6','MAF','GFI1','IRF4','NOTCH1','NOTCH2','IL1RL1','IL17RB','IFNGR1','IFNGR2','TNFRSF8',#Th2
                      'IL2','IL5','IL6','IL10','IL13','IL31',#Th2分泌
                      'CCR4','CCR6','CCR10','AHR','PDGFRA','PDGFRB',#Th22
                      'IL22','TNF',#Th22分泌
                      'CXCR3','CCR5','KLRD1','TBX21','STAT1','STAT4','EOMES','RUNX3','FASLG','IL12RB1','IL12RB2','IL18R1','IL27RA','NOTCH3','TNFSF11','ICOS','HAVCR2','DPP4',#Th1
                      'LTB','LTA','PRF1','GZMB','GZMA','TNF','IFNG',#Th1分泌
                      'CCR4','CCR6','KLRB1','ICOS','HAVCR2','RORC','RORA','STAT3','RUNX1','BATF','IRF4','MAF','IL6R','IL13RA1','IL21R','IL23R',#Th17
                      'TNF','CCL20','IL17A','IL17F','IL21','IL22','IL24','IL26',#Th17分泌
                      ],
              standard_scale='obs',groupby=groupby)

In [ ]:
adata.obs['cluster_dummy']="NOT"
adata.obs.loc[adata.obs[groupby]=="6",'cluster_dummy'] = "YES"
sc.pl.umap(adata, color='cluster_dummy')

In [87]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.5
0    35784
1    32193
3    27352
2    18252
Name: count, dtype: int64

In [88]:
#Naïve CD4+ T 需要CD45RA+ CCR7=CD197hi CD62Lhi
# effector memory T cells(TEM, CD45RA-/CCR7-)
# TEMRA cells, which are T cells that re-express CD45RA(CD45RA+/CCR7-)

#'Th22':#CCR4+ CCR6+ CCR10+ KLRB1- GZMK- GATA3lo CCL5- GATA3+,CCR4+ CCR6+ KLRB1- GZMKlo AHRlo
#'Th1/Th17':# CCR6+ CCR4- CXCR3+ KLRB1+ GZMK+ CCL5+
#'Th17':#step1 CCR6+ CCR4- KLRB1+ GZMK- CCL5- RORC+ GZMK- TBX21-
#'Th2':#step2 CCR4+ CCR6- KLRB1- GATA3+ GZMK- CCL5-；CCR4+ CXCR5- GATA3+ CCL5-.CD45RA- CD279- TBX21- LEF1+ ,且没有CCL5- GZMH- GZMK- GZMB- KLRB1-各种标记的情况下CD62L+ CD27+ CD25+ 
#'Th1':#step3 确认 CXCR3+ CCR6- KLRB1- GZMK+ CD279+ GZMK+ CCL5+. CD279+(核心) GZMK+(核心) TBX21+ CCL5+(核心) TIGIT+ KLRB1-(核心) CD127-/IL7Rlo CCR7-CD197- SELLlo/CD62Llo  GZMH- GNLY- PRF1- GZMB- 的是Th1

cell_dict = {#'Th22':[],#CCR4+ CCR6+ CCR10+ KLRB1lo/- GZMK- AHR+   CRIP1 LGALS1 LGALS3 PI16 ANXA5 ANXA2   https://www.sciencedirect.com/science/article/pii/S1933021922002033
             #'Th2|Th22':[''],#PTGDR2=CRTH2 GATA3++ IL4R+ CCR4+ CCR6- KLRB1- GZMK- CCL5- CD62L+ CD25+     PTGDR2,SNED1,NEFL,GATA3,FXYD7,C1orf162
             'CD27+ Th1':['0','1',],#EOMES+ GZMK+ CCL5+ KLRB1- CXCR3+ CCR6- CD279+ TBX21+ IFNG+.  CMC1,CST7,FCRL3,CCL4,SLAMF7,EOMES,PDCD1,NKG7,CCR5,KLRK1,F2R,PLEK
             'CD27- Th1':['2','3',],#EOMES+ GZMK+ CCL5+ KLRB1- CXCR3+ CCR6- CD279+ TBX21+ IFNG+.  CMC1,CST7,FCRL3,CCL4,SLAMF7,EOMES,PDCD1,NKG7,CCR5,KLRK1,F2R,PLEK
             #'Th17':[],#CCR6=CD196++ RORC+ CCR4- KLRB1+ GZMK- CCL5- ICOS+    高表达LTK,PTPN13,PDE4D,CCR6,RORC,NR1D1,CTSH,KIF5C,LGALS3,USP10,CMTM6,TOB1
             #'Th1/Th17':[],#DPP4+ CCR6+ EOMESlo CCR6lo CCR4- CXCR3+ KLRB1+ GZMKlo/+ CCL5+ TBX21+ 与Th1相似, 差异基因中等表达 https://rupress.org/jem/article/211/1/89/41376/Pro-inflammatory-human-Th17-cells-selectively
             #'HLA-DRhi memory':[,],#增殖特性
            }

In [89]:
check_dict_duplicates(cell_dict)

'无重复'

In [90]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R5'] = i

In [91]:
(adata.obs['Celltype_L4_L5_Refine_R5'].isna()).value_counts()

Celltype_L4_L5_Refine_R5
False    113581
Name: count, dtype: int64

In [92]:
adata.obs['Celltype_L4_L5_Refine_R5'].value_counts()

Celltype_L4_L5_Refine_R5
CD27+ Th1    67977
CD27- Th1    45604
Name: count, dtype: int64

In [93]:
adata.obs['Celltype_L4_L5_Refine_R5'].value_counts()

Celltype_L4_L5_Refine_R5
CD27+ Th1    67977
CD27- Th1    45604
Name: count, dtype: int64

In [95]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine',
                           'Celltype_L3_L4_Refine','Celltype_L4_L5_Refine',
                           'Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3',
                           'Celltype_L4_L5_Refine_R4','Celltype_L4_L5_Refine_R5',
                           groupby,'receptor_type','receptor_type_BCR']]
indices['UMAP_1'] = adata.obsm['X_umap'][:, 0].copy()  #
indices['UMAP_2'] = adata.obsm['X_umap'][:, 1].copy()  #
indices.rename(columns={groupby: 'leiden_cluster'}, inplace=True) #Save the cluster categorical, check the relationship between clusters and clinical to avoid missing someone.
indices['leiden_cluster'] = celltype + " c" + indices['leiden_cluster'].astype(str)
os.makedirs(f'{finnal_path}/{celltype}', exist_ok=True)
indices.to_csv(f"{finnal_path}/{celltype}/Finnal_indices_{celltype}.csv")
indices.head()

,Celltype_L1_L2,Celltype_L1_L2_Refine,Celltype_L2_L3_Refine,Celltype_L3_L4_Refine,Celltype_L4_L5_Refine,Celltype_L4_L5_Refine_R2,Celltype_L4_L5_Refine_R3,Celltype_L4_L5_Refine_R4,Celltype_L4_L5_Refine_R5,leiden_cluster,receptor_type,receptor_type_BCR,UMAP_1,UMAP_2
D0908_Rep2_TAGGATGA_TCTTCACA_ATTGGCTC,CD4+ T,Naïve CD4+ T,Treg,Tem CD4+ T(Treg),Treg memory CD4+ T,Th17(fromTreg),Th1,Th1,CD27+ Th1,Th1 c0,TCR,NaN,7.874675,7.908182
D0109_Rep1_ATTGAGGA_GTCGTAGA_TGGCTTCA,CD4+ T,Naïve CD4+ T,Treg,Treg CD4+,Treg Naive CD4+ T,Th2(fromTreg),Th1,Th1,CD27+ Th1,Th1 c0,TCR,NaN,7.523333,8.206389
D0502_Rep2_CTGGCATA_AGATCGCA_TGGAACAA,CD4+ T,Naïve CD4+ T,Treg,Treg CD4+,Treg memory CD4+ T,Th17(fromTreg),Th1,Th1,CD27+ Th1,Th1 c0,NaN,NaN,8.634336,8.658976
D0642_E_Rep2_CGGATTGC_TATCAGCA_CGAACTTA,CD4+ T,Naïve CD4+ T,Treg,Tem CD4+ T(Treg),Treg memory CD4+ T,Th17(fromTreg),Th1,Th1,CD27+ Th1,Th1 c1,TCR,NaN,6.554946,6.023137
D0851_Rep1_ACACAGAA_GTGTTCTA_AGTACAAG,CD4+ T,Naïve CD4+ T,Treg,Tem CD4+ T(Treg),Treg memory CD4+ T,Th17(fromTreg),Th1,Th1,CD27+ Th1,Th1 c1,TCR,NaN,9.361308,3.631063


# 2. 定 终 Th22 T Cell Refine

In [96]:
celltype="Th22"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [97]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L4_L5_Refine_R3','Celltype_L4_L5_Refine_R4'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4,
           )

In [98]:
groupby = "L4_leiden_TOTALVI_0.5"

In [366]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [188]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='obs',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
sc.pl.umap(adt, color='HLA-DR',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',layer='dsb')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='CD183',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD185',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False,layer='dsb')
sc.pl.umap(adt, color='CD196',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False,layer='dsb')
sc.pl.umap(adt, color='CD161',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False,layer='dsb')
sc.pl.umap(adt, color='CD45RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD16',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False,layer='dsb')
sc.pl.umap(adt, color='IgM',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False,layer='dsb')
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='Celltype_L4_L5_Refine_R2',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='KLRB1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='IFNG',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='CXCR3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='CCR6',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.umap(adata, color='CD27',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',size=2)

In [ ]:
sc.pl.umap(adata, color='CD27',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',size=2)

In [ ]:
#Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','AHR',
                      'LTK','PTPN13','PDE4D','CCR6','RORC','NR1D1','CTSH','KIF5C','LGALS3','USP10','CMTM6','TOB1',
                     'TNFSF13B','CISH','AQP3','AUTS2','NSG1','S100A4'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1/Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'GZMH','IL18RAP','S1PR5','LYAR','NKG7','CST7','PRF1','TBX21','LINC01871','KLRG1','MYBL1','EOMES',
                     'EFHD2','DUSP2','SAMD3','CTSW','ID2','MATK','HOPX',],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','TBX21','IFNG',
                      'CMC1','CST7','FCRL3','CCL4','SLAMF7','EOMES','PDCD1','NKG7','CCR5','KLRK1','F2R','PLEK'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th22
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','AHR',
                      'CRIP1','LGALS1','LGALS1','S100A10','S100A4','PI16','LMNA','ANXA5','ANXA2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th2
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'PTGDR2','SNED1','NEFL','GATA3','FXYD7','C1orf162','GDPD5','IL4R','CAPG',
                     'LGALS1','TNFSF10','TNFRSF4','PPP1R9B','CSGALNACT1','NIBAN1','ERN1','SORL1','RUNX2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','LAG3','HAVCR2','BTBD9',
                      'CCL5','CTLA4','CD40LG'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th from BD
#TNFSF8=CD30L B3GAT1=CD57  BTLA= CD272  SLAMF5=CD84 HAVCR2=CD365
sc.pl.dotplot(adata, ['CXCR5','IL6R','TNFSF8','NRP1','IL21R','B3GAT1','BCL6','MAF','STAT3','ICOS','PDCD1','TIGIT','BTLA','CD200','SLAMF1','CD84',#Tfh
                      'IL4','IL17F','IL17A','IL21',#tfh分泌
                      'GATA3','SMAD1','STAT6','SPI1','IRF4',#Th9
                      'IL9','IL10','CCL17','CCL22','TGFB1',#th9分泌
                      'HAVCR2','CXCR4','CCR3','CCR4','CCR8','PTGDR2','GATA3','STAT5A','STAT6','MAF','GFI1','IRF4','NOTCH1','NOTCH2','IL1RL1','IL17RB','IFNGR1','IFNGR2','TNFRSF8',#Th2
                      'IL2','IL5','IL6','IL10','IL13','IL31',#Th2分泌
                      'CCR4','CCR6','CCR10','AHR','PDGFRA','PDGFRB',#Th22
                      'IL22','TNF',#Th22分泌
                      'CXCR3','CCR5','KLRD1','TBX21','STAT1','STAT4','EOMES','RUNX3','FASLG','IL12RB1','IL12RB2','IL18R1','IL27RA','NOTCH3','TNFSF11','ICOS','HAVCR2','DPP4',#Th1
                      'LTB','LTA','PRF1','GZMB','GZMA','TNF','IFNG',#Th1分泌
                      'CCR4','CCR6','KLRB1','ICOS','HAVCR2','RORC','RORA','STAT3','RUNX1','BATF','IRF4','MAF','IL6R','IL13RA1','IL21R','IL23R',#Th17
                      'TNF','CCL20','IL17A','IL17F','IL21','IL22','IL24','IL26',#Th17分泌
                      ],
              standard_scale='obs',groupby=groupby)

In [ ]:
adata.obs['cluster_dummy']="NOT"
adata.obs.loc[adata.obs[groupby]=="6",'cluster_dummy'] = "YES"
sc.pl.umap(adata, color='cluster_dummy')

In [99]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.5
1    22602
2    21870
0    19757
3     3181
4     2113
Name: count, dtype: int64

In [100]:
#Naïve CD4+ T 需要CD45RA+ CCR7=CD197hi CD62Lhi
# effector memory T cells(TEM, CD45RA-/CCR7-)
# TEMRA cells, which are T cells that re-express CD45RA(CD45RA+/CCR7-)

#'Th22':#CCR4+ CCR6+ CCR10+ KLRB1- GZMK- GATA3lo CCL5- GATA3+,CCR4+ CCR6+ KLRB1- GZMKlo AHRlo
#'Th1/Th17':# CCR6+ CCR4- CXCR3+ KLRB1+ GZMK+ CCL5+
#'Th17':#step1 CCR6+ CCR4- KLRB1+ GZMK- CCL5- RORC+ GZMK- TBX21-
#'Th2':#step1 CCR4+ CCR6- KLRB1- GATA3+ GZMK- CCL5-；CCR4+ CXCR5- GATA3+ CCL5-.CD45RA- CD279- TBX21- LEF1+ ,且没有CCL5- GZMH- GZMK- GZMB- KLRB1-各种标记的情况下CD62L+ CD27+ CD25+ 
#'Th1':#step3 确认 CXCR3+ CCR6- KLRB1- GZMK+ CD279+ GZMK+ CCL5+. CD279+(核心) GZMK+(核心) TBX21+ CCL5+(核心) TIGIT+ KLRB1-(核心) CD127-/IL7Rlo CCR7-CD197- SELLlo/CD62Llo  GZMH- GNLY- PRF1- GZMB- 的是Th1

cell_dict = {'Th22':['0','1','2',],#CCR4+ CCR6+ CCR10+ KLRB1lo/- GZMK-    CRIP1 LGALS1 LGALS3 PI16 ANXA5 ANXA2
             #'Th2':[],#PTGDR2=CRTH2 GATA3++ IL4R+ CCR4+ CCR6- KLRB1- GZMK- CCL5- CD62L+ CD25+     PTGDR2,SNED1,NEFL,GATA3,FXYD7,C1orf162
             #'Th1':[],#EOMES+ GZMK+ CCL5+ KLRB1- CXCR3+ CCR6- CD279+ TBX21+ IFNG+.  CMC1,CST7,FCRL3,CCL4,SLAMF7,EOMES,PDCD1,NKG7,CCR5,KLRK1,F2R,PLEK
             #'Th17':[''],#CCR6=CD196++ RORC+ CCR4- KLRB1+ GZMK- CCL5- ICOS+    高表达LTK,PTPN13,PDE4D,CCR6,RORC,NR1D1,CTSH,KIF5C,LGALS3,USP10,CMTM6,TOB1
             #'Th1/Th17':['',],#DPP4+ CCR6+ EOMESlo CCR6lo CCR4- CXCR3+ KLRB1+ GZMKlo/+ CCL5+ TBX21+ 与Th1相似, 差异基因中等表达
             'Doublet|Lowquality':['4',
                                  '3'],#CCR10 IFNG
            }

In [101]:
check_dict_duplicates(cell_dict)

'无重复'

In [102]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R5'] = i

In [103]:
(adata.obs['Celltype_L4_L5_Refine_R5'].isna()).value_counts()

Celltype_L4_L5_Refine_R5
False    69523
Name: count, dtype: int64

In [104]:
adata.obs['Celltype_L4_L5_Refine_R5'].value_counts()

Celltype_L4_L5_Refine_R5
Th22                  64229
Doublet|Lowquality     5294
Name: count, dtype: int64

In [105]:
adata = adata[adata.obs['Celltype_L4_L5_Refine_R5'] != "Doublet|Lowquality",:]

In [106]:
adata.obs['Celltype_L4_L5_Refine_R5'].value_counts()

Celltype_L4_L5_Refine_R5
Th22    64229
Name: count, dtype: int64

In [107]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine',
                           'Celltype_L3_L4_Refine','Celltype_L4_L5_Refine',
                           'Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3',
                           'Celltype_L4_L5_Refine_R4','Celltype_L4_L5_Refine_R5',
                           groupby,'receptor_type','receptor_type_BCR']]
indices['UMAP_1'] = adata.obsm['X_umap'][:, 0].copy()  #
indices['UMAP_2'] = adata.obsm['X_umap'][:, 1].copy()  #
indices.rename(columns={groupby: 'leiden_cluster'}, inplace=True) #Save the cluster categorical, check the relationship between clusters and clinical to avoid missing someone.
indices['leiden_cluster'] = celltype + " c" + indices['leiden_cluster'].astype(str)
os.makedirs(f'{finnal_path}/{celltype}', exist_ok=True)
indices.to_csv(f"{finnal_path}/{celltype}/Finnal_indices_{celltype}.csv")
indices.head()

,Celltype_L1_L2,Celltype_L1_L2_Refine,Celltype_L2_L3_Refine,Celltype_L3_L4_Refine,Celltype_L4_L5_Refine,Celltype_L4_L5_Refine_R2,Celltype_L4_L5_Refine_R3,Celltype_L4_L5_Refine_R4,Celltype_L4_L5_Refine_R5,leiden_cluster,receptor_type,receptor_type_BCR,UMAP_1,UMAP_2
D0027_Rep2_CGCTGATC_AACAACCA_CTGAGCCA,CD4+ T,Naïve CD4+ T,Treg,Treg CD4+,Treg memory CD4+ T,Th17(fromTreg),Th1,Th22,Th22,Th22 c0,TCR,NaN,4.562911,1.936775
D0864_M_Rep2_CCATCCTC_GAATCTGA_CCTAATCC,CD4+ T,Treg,Treg,Treg CD4+,Treg memory CD4+ T,Th2(fromTreg),Th1,Th22,Th22,Th22 c1,TCR,NaN,5.301332,4.610394
D0428_Rep2_GAGTTAGC_AAGACGGA_CCTCTATC,CD4+ T,Treg,Treg,Treg CD4+,Treg memory CD4+ T,Th2(fromTreg),Th1,Th22,Th22,Th22 c1,TCR,NaN,6.399027,4.596732
D0428_Rep2_AGCACCTC_TGGTGGTA_AAACATCG,CD4+ T,Treg,Treg,Treg CD4+,Treg memory CD4+ T,Th2(fromTreg),Th1,Th22,Th22,Th22 c1,TCR,NaN,4.065114,4.308391
D0130_Rep1_GAGCTGAA_AGTACAAG_ACGCTCGA,CD4+ T,Treg,Treg,Treg CD4+,Treg memory CD4+ T,Th2(fromTreg),Th1,Th22,Th22,Th22 c1,TCR,NaN,5.747772,6.068399


# 3. 定 终 Th2 T Cell Refine

In [108]:
celltype="Th2"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [109]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L4_L5_Refine_R3','Celltype_L4_L5_Refine_R4'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4,
           )

In [110]:
groupby = "L4_leiden_TOTALVI_0.5"

In [339]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [341]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='obs',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
sc.pl.umap(adt, color='HLA-DR',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',layer='dsb')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='CD183',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD185',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False,layer='dsb')
sc.pl.umap(adt, color='CD196',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False,layer='dsb')
sc.pl.umap(adt, color='CD161',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False,layer='dsb')
sc.pl.umap(adt, color='CD45RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD16',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False,layer='dsb')
sc.pl.umap(adt, color='IgM',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False,layer='dsb')
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='Celltype_L4_L5_Refine_R2',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='KLRB1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='IFNG',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='CXCR3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='CCR6',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',size=2)

In [ ]:
sc.pl.umap(adata, color='CCR10',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',size=2)

In [ ]:
#Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','AHR',
                      'LTK','PTPN13','PDE4D','CCR6','RORC','NR1D1','CTSH','KIF5C','LGALS3','USP10','CMTM6','TOB1',
                     'TNFSF13B','CISH','AQP3','AUTS2','NSG1','S100A4'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1/Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'GZMH','IL18RAP','S1PR5','LYAR','NKG7','CST7','PRF1','TBX21','LINC01871','KLRG1','MYBL1','EOMES',
                     'EFHD2','DUSP2','SAMD3','CTSW','ID2','MATK','HOPX',],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','TBX21','IFNG',
                      'CMC1','CST7','FCRL3','CCL4','SLAMF7','EOMES','PDCD1','NKG7','CCR5','KLRK1','F2R','PLEK'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th22
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'CRIP1','LGALS1','LGALS1','S100A10','S100A4','PI16','LMNA','ANXA5','ANXA2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th2
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'PTGDR2','SNED1','NEFL','GATA3','FXYD7','C1orf162','GDPD5','IL4R','CAPG',
                     'LGALS1','TNFSF10','TNFRSF4','PPP1R9B','CSGALNACT1','NIBAN1','ERN1','SORL1','RUNX2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','LAG3','HAVCR2','BTBD9',
                      'CCL5','CTLA4','CD40LG'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th from BD
#TNFSF8=CD30L B3GAT1=CD57  BTLA= CD272  SLAMF5=CD84 HAVCR2=CD365
sc.pl.dotplot(adata, ['CXCR5','IL6R','TNFSF8','NRP1','IL21R','B3GAT1','BCL6','MAF','STAT3','ICOS','PDCD1','TIGIT','BTLA','CD200','SLAMF1','CD84',#Tfh
                      'IL4','IL17F','IL17A','IL21',#tfh分泌
                      'GATA3','SMAD1','STAT6','SPI1','IRF4',#Th9
                      'IL9','IL10','CCL17','CCL22','TGFB1',#th9分泌
                      'HAVCR2','CXCR4','CCR3','CCR4','CCR8','PTGDR2','GATA3','STAT5A','STAT6','MAF','GFI1','IRF4','NOTCH1','NOTCH2','IL1RL1','IL17RB','IFNGR1','IFNGR2','TNFRSF8',#Th2
                      'IL2','IL5','IL6','IL10','IL13','IL31',#Th2分泌
                      'CCR4','CCR6','CCR10','AHR','PDGFRA','PDGFRB',#Th22
                      'IL22','TNF',#Th22分泌
                      'CXCR3','CCR5','KLRD1','TBX21','STAT1','STAT4','EOMES','RUNX3','FASLG','IL12RB1','IL12RB2','IL18R1','IL27RA','NOTCH3','TNFSF11','ICOS','HAVCR2','DPP4',#Th1
                      'LTB','LTA','PRF1','GZMB','GZMA','TNF','IFNG',#Th1分泌
                      'CCR4','CCR6','KLRB1','ICOS','HAVCR2','RORC','RORA','STAT3','RUNX1','BATF','IRF4','MAF','IL6R','IL13RA1','IL21R','IL23R',#Th17
                      'TNF','CCL20','IL17A','IL17F','IL21','IL22','IL24','IL26',#Th17分泌
                      ],
              standard_scale='obs',groupby=groupby)

In [ ]:
adata.obs['cluster_dummy']="NOT"
adata.obs.loc[adata.obs[groupby]=="5",'cluster_dummy'] = "YES"
sc.pl.umap(adata, color='cluster_dummy')

In [111]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.5
2    35655
1    26989
0    23927
Name: count, dtype: int64

In [112]:
#Naïve CD4+ T 需要CD45RA+ CCR7=CD197hi CD62Lhi
# effector memory T cells(TEM, CD45RA-/CCR7-)
# TEMRA cells, which are T cells that re-express CD45RA(CD45RA+/CCR7-)

#'Th22':#CCR4+ CCR6+ CCR10+ KLRB1- GZMK- GATA3lo CCL5- GATA3+,CCR4+ CCR6+ KLRB1- GZMKlo AHRlo
#'Th1/Th17':# CCR6+ CCR4- CXCR3+ KLRB1+ GZMK+ CCL5+
#'Th17':#step1 CCR6+ CCR4- KLRB1+ GZMK- CCL5- RORC+ GZMK- TBX21-
#'Th2':#step1 CCR4+ CCR6- KLRB1- GATA3+ GZMK- CCL5-；CCR4+ CXCR5- GATA3+ CCL5-.CD45RA- CD279- TBX21- LEF1+ ,且没有CCL5- GZMH- GZMK- GZMB- KLRB1-各种标记的情况下CD62L+ CD27+ CD25+ 
#'Th1':#step3 确认 CXCR3+ CCR6- KLRB1- GZMK+ CD279+ GZMK+ CCL5+. CD279+(核心) GZMK+(核心) TBX21+ CCL5+(核心) TIGIT+ KLRB1-(核心) CD127-/IL7Rlo CCR7-CD197- SELLlo/CD62Llo  GZMH- GNLY- PRF1- GZMB- 的是Th1

cell_dict = {#'Th22':[],#CCR4+ CCR6+ CCR10+ KLRB1lo/- GZMK-    CRIP1 LGALS1 LGALS3 PI16 ANXA5 ANXA2
             'Th2':['0','1','2'],#PTGDR2=CRTH2 GATA3++ IL4R+ CCR4+ CCR6- KLRB1- GZMK- CCL5- CD62L+ CD25+     PTGDR2,SNED1,NEFL,GATA3,FXYD7,C1orf162
             #'Th1':['2',],#EOMES+ GZMK+ CCL5+ KLRB1- CXCR3+ CCR6- CD279+ TBX21+ IFNG+.  CMC1,CST7,FCRL3,CCL4,SLAMF7,EOMES,PDCD1,NKG7,CCR5,KLRK1,F2R,PLEK
             #'Th17':[],#CCR6=CD196++ RORC+ CCR4- KLRB1+ GZMK- CCL5- ICOS+    高表达LTK,PTPN13,PDE4D,CCR6,RORC,NR1D1,CTSH,KIF5C,LGALS3,USP10,CMTM6,TOB1
             #'Th1/Th17':['',],#DPP4+ CCR6+ EOMESlo CCR6lo CCR4- CXCR3+ KLRB1+ GZMKlo/+ CCL5+ TBX21+ 与Th1相似, 差异基因中等表达
             #'Doublet|Lowquality':[],
            }

In [113]:
check_dict_duplicates(cell_dict)

'无重复'

In [114]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R5'] = i

In [115]:
(adata.obs['Celltype_L4_L5_Refine_R5'].isna()).value_counts()

Celltype_L4_L5_Refine_R5
False    86571
Name: count, dtype: int64

In [116]:
adata.obs['Celltype_L4_L5_Refine_R5'].value_counts()

Celltype_L4_L5_Refine_R5
Th2    86571
Name: count, dtype: int64

In [117]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine',
                           'Celltype_L3_L4_Refine','Celltype_L4_L5_Refine',
                           'Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3',
                           'Celltype_L4_L5_Refine_R4','Celltype_L4_L5_Refine_R5',
                           groupby,'receptor_type','receptor_type_BCR']]
indices['UMAP_1'] = adata.obsm['X_umap'][:, 0].copy()  #
indices['UMAP_2'] = adata.obsm['X_umap'][:, 1].copy()  #
indices.rename(columns={groupby: 'leiden_cluster'}, inplace=True) #Save the cluster categorical, check the relationship between clusters and clinical to avoid missing someone.
indices['leiden_cluster'] = celltype + " c" + indices['leiden_cluster'].astype(str)
os.makedirs(f'{finnal_path}/{celltype}', exist_ok=True)
indices.to_csv(f"{finnal_path}/{celltype}/Finnal_indices_{celltype}.csv")
indices.head()

,Celltype_L1_L2,Celltype_L1_L2_Refine,Celltype_L2_L3_Refine,Celltype_L3_L4_Refine,Celltype_L4_L5_Refine,Celltype_L4_L5_Refine_R2,Celltype_L4_L5_Refine_R3,Celltype_L4_L5_Refine_R4,Celltype_L4_L5_Refine_R5,leiden_cluster,receptor_type,receptor_type_BCR,UMAP_1,UMAP_2
D0428_Rep2_CAATGGAA_CACCTTAC_AAGGTACA,CD4+ T,Naïve CD4+ T,Treg,Treg CD4+,Treg memory CD4+ T,Th2(fromTreg),Th2|Th22,Th2,Th2,Th2 c0,TCR,NaN,6.306913,8.610665
D0428_Rep2_AGATGTAC_CATACCAA_AGATCGCA,CD4+ T,Naïve CD4+ T,Treg,Tem CD4+ T(Treg),Treg memory CD4+ T,Th2(fromTreg),Th2|Th22,Th2,Th2,Th2 c1,TCR,NaN,-0.911757,6.931706
D0130_Rep1_ACACAGAA_GCGAGTAA_ACACAGAA,CD4+ T,Naïve CD4+ T,Treg,Tem CD4+ T(Treg),Treg memory CD4+ T,Th2(fromTreg),Th2|Th22,Th2,Th2,Th2 c0,TCR,NaN,5.259487,8.762287
D0615_Rep1_CCATCCTC_AAGACGGA_AAGACGGA,CD4+ T,Naïve CD4+ T,Treg,Tem CD4+ T(Treg),Treg memory CD4+ T,Th2(fromTreg),Th2|Th22,Th2,Th2,Th2 c1,TCR,NaN,4.864313,7.562672
D0629_E_Rep1_TGAAGAGA_AAGACGGA_CACCTTAC,CD4+ T,Naïve CD4+ T,Treg,Tem CD4+ T(Treg),Treg memory CD4+ T,Th2(fromTreg),Th2|Th22,Th2,Th2,Th2 c1,TCR,NaN,0.266920,8.966625


# 4. 定 终 Th17 T Cell Refine

In [118]:
celltype="Th17"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [ ]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4,
           )

In [ ]:
groupby = "L4_leiden_TOTALVI_0.5"

In [285]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [287]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='obs',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
sc.pl.umap(adt, color='HLA-DR',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',layer='dsb')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='CD183',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD185',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False,layer='dsb')
sc.pl.umap(adt, color='CD196',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False,layer='dsb')
sc.pl.umap(adt, color='CD161',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False,layer='dsb')
sc.pl.umap(adt, color='CD45RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD16',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False,layer='dsb')
sc.pl.umap(adt, color='IgM',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False,layer='dsb')
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='Celltype_L4_L5_Refine_R2',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='KLRB1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='IFNG',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='CXCR3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='CCR6',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.umap(adata, color='CD27',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',size=2)

In [ ]:
sc.pl.umap(adata, color='FOXP3',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',size=2)

In [ ]:
#Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','CD27',
                      'LTK','PTPN13','PDE4D','CCR6','RORC','NR1D1','CTSH','KIF5C','LGALS3','USP10','CMTM6','TOB1','MAF',
                     'TNFSF13B','CISH','AQP3','AUTS2','NSG1','S100A4'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1/Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'GZMH','IL18RAP','S1PR5','LYAR','NKG7','CST7','PRF1','TBX21','LINC01871','KLRG1','MYBL1','EOMES',
                     'EFHD2','DUSP2','SAMD3','CTSW','ID2','MATK','HOPX',],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','TBX21','IFNG',
                      'CMC1','CST7','FCRL3','CCL4','SLAMF7','EOMES','PDCD1','NKG7','CCR5','KLRK1','F2R','PLEK'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th22
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','AHR',
                      'CRIP1','LGALS1','LGALS1','S100A10','S100A4','PI16','LMNA','ANXA5','ANXA2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th2
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'PTGDR2','SNED1','NEFL','GATA3','FXYD7','C1orf162','GDPD5','IL4R','CAPG',
                     'LGALS1','TNFSF10','TNFRSF4','PPP1R9B','CSGALNACT1','NIBAN1','ERN1','SORL1','RUNX2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','LAG3','HAVCR2','BTBD9',
                      'CCL5','CTLA4','CD40LG'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th from BD
#TNFSF8=CD30L B3GAT1=CD57  BTLA= CD272  SLAMF5=CD84 HAVCR2=CD365
sc.pl.dotplot(adata, ['CXCR5','IL6R','TNFSF8','NRP1','IL21R','B3GAT1','BCL6','MAF','STAT3','ICOS','PDCD1','TIGIT','BTLA','CD200','SLAMF1','CD84',#Tfh
                      'IL4','IL17F','IL17A','IL21',#tfh分泌
                      'GATA3','SMAD1','STAT6','SPI1','IRF4',#Th9
                      'IL9','IL10','CCL17','CCL22','TGFB1',#th9分泌
                      'HAVCR2','CXCR4','CCR3','CCR4','CCR8','PTGDR2','GATA3','STAT5A','STAT6','MAF','GFI1','IRF4','NOTCH1','NOTCH2','IL1RL1','IL17RB','IFNGR1','IFNGR2','TNFRSF8',#Th2
                      'IL2','IL5','IL6','IL10','IL13','IL31',#Th2分泌
                      'CCR4','CCR6','CCR10','AHR','PDGFRA','PDGFRB',#Th22
                      'IL22','TNF',#Th22分泌
                      'CXCR3','CCR5','KLRD1','TBX21','STAT1','STAT4','EOMES','RUNX3','FASLG','IL12RB1','IL12RB2','IL18R1','IL27RA','NOTCH3','TNFSF11','ICOS','HAVCR2','DPP4',#Th1
                      'LTB','LTA','PRF1','GZMB','GZMA','TNF','IFNG',#Th1分泌
                      'CCR4','CCR6','KLRB1','ICOS','HAVCR2','RORC','RORA','STAT3','RUNX1','BATF','IRF4','MAF','IL6R','IL13RA1','IL21R','IL23R',#Th17
                      'TNF','CCL20','IL17A','IL17F','IL21','IL22','IL24','IL26',#Th17分泌
                      ],
              standard_scale='obs',groupby=groupby)

In [ ]:
adata.obs['cluster_dummy']="NOT"
adata.obs.loc[adata.obs[groupby]=="6",'cluster_dummy'] = "YES"
sc.pl.umap(adata, color='cluster_dummy')

In [ ]:
adata.obs[groupby].value_counts()

In [ ]:
#Naïve CD4+ T 需要CD45RA+ CCR7=CD197hi CD62Lhi
# effector memory T cells(TEM, CD45RA-/CCR7-)
# TEMRA cells, which are T cells that re-express CD45RA(CD45RA+/CCR7-)

#'Th22':#CCR4+ CCR6+ CCR10+ KLRB1- GZMK- GATA3lo CCL5- GATA3+,CCR4+ CCR6+ KLRB1- GZMKlo AHRlo
#'Th1/Th17':# CCR6+ CCR4- CXCR3+ KLRB1+ GZMK+ CCL5+
#'Th17':#step1 CCR6+ CCR4- KLRB1+ GZMK- CCL5- RORC+ GZMK- TBX21-
#'Th2':#step1 CCR4+ CCR6- KLRB1- GATA3+ GZMK- CCL5-；CCR4+ CXCR5- GATA3+ CCL5-.CD45RA- CD279- TBX21- LEF1+ ,且没有CCL5- GZMH- GZMK- GZMB- KLRB1-各种标记的情况下CD62L+ CD27+ CD25+ 
#'Th1':#step3 确认 CXCR3+ CCR6- KLRB1- GZMK+ CD279+ GZMK+ CCL5+. CD279+(核心) GZMK+(核心) TBX21+ CCL5+(核心) TIGIT+ KLRB1-(核心) CD127-/IL7Rlo CCR7-CD197- SELLlo/CD62Llo  GZMH- GNLY- PRF1- GZMB- 的是Th1

cell_dict = {#'Th22':['',],#CCR4+ CCR6+ CCR10+ KLRB1lo/- GZMK-    CRIP1 LGALS1 LGALS3 PI16 ANXA5 ANXA2
             #'Th2|Th22':[''],#PTGDR2=CRTH2 GATA3++ IL4R+ CCR4+ CCR6- KLRB1- GZMK- CCL5- CD62L+ CD25+     PTGDR2,SNED1,NEFL,GATA3,FXYD7,C1orf162
             #'Th1':[,],#EOMES+ GZMK+ CCL5+ KLRB1- CXCR3+ CCR6- CD279+ TBX21+ IFNG+.  CMC1,CST7,FCRL3,CCL4,SLAMF7,EOMES,PDCD1,NKG7,CCR5,KLRK1,F2R,PLEK
             'CD27+ Th17':['0','1','2','5','6','7'],#CCR6=CD196++ RORC+ CCR4- KLRB1+ GZMK- CCL5- ICOS+    高表达LTK,PTPN13,PDE4D,CCR6,RORC,NR1D1,CTSH,KIF5C,LGALS3,USP10,CMTM6,TOB1
             'CD27- Th17':['3'],#CCR6=CD196++ RORC+ CCR4- KLRB1+ GZMK- CCL5- ICOS+    高表达LTK,PTPN13,PDE4D,CCR6,RORC,NR1D1,CTSH,KIF5C,LGALS3,USP10,CMTM6,TOB1
             #'Th1/Th17':['',],#DPP4+ CCR6+ EOMESlo CCR6lo CCR4- CXCR3+ KLRB1+ GZMKlo/+ CCL5+ TBX21+ 与Th1相似, 差异基因中等表达
             'Doublet|Lowquality':['4'],
            }

In [ ]:
check_dict_duplicates(cell_dict)

In [ ]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R5'] = i

In [ ]:
(adata.obs['Celltype_L4_L5_Refine_R5'].isna()).value_counts()

In [ ]:
adata.obs['Celltype_L4_L5_Refine_R5'].value_counts()

In [ ]:
adata = adata[adata.obs['Celltype_L4_L5_Refine_R5'] != "Doublet|Lowquality",:]

In [ ]:
adata.obs['Celltype_L4_L5_Refine_R5'].value_counts()

In [ ]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine',
                           'Celltype_L3_L4_Refine','Celltype_L4_L5_Refine',
                           'Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3',
                           'Celltype_L4_L5_Refine_R4','Celltype_L4_L5_Refine_R5',
                           groupby,'receptor_type','receptor_type_BCR']]
indices['UMAP_1'] = adata.obsm['X_umap'][:, 0].copy()  #
indices['UMAP_2'] = adata.obsm['X_umap'][:, 1].copy()  #
indices.rename(columns={groupby: 'leiden_cluster'}, inplace=True) #Save the cluster categorical, check the relationship between clusters and clinical to avoid missing someone.
indices['leiden_cluster'] = celltype + " c" + indices['leiden_cluster'].astype(str)
os.makedirs(f'{finnal_path}/{celltype}', exist_ok=True)
indices.to_csv(f"{finnal_path}/{celltype}/Finnal_indices_{celltype}.csv")
indices.head()

# 5. 定 终 Th1_Th17 T Cell Refine. Although we obtained the final annotation, it still includes other T cell subsets Tfh/Tcm, Th17, and Th1.

In [ ]:
celltype="Th1_Th17"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [ ]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3','Celltype_L4_L5_Refine_R4',],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4,
           )

In [ ]:
groupby = "L4_leiden_TOTALVI_1.5"

In [540]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [542]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='obs',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
sc.pl.umap(adt, color='HLA-DR',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',layer='dsb')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='CD183',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD185',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False,layer='dsb')
sc.pl.umap(adt, color='CD196',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False,layer='dsb')
sc.pl.umap(adt, color='CD161',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False,layer='dsb')
sc.pl.umap(adt, color='CD45RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD16',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False,layer='dsb')
sc.pl.umap(adt, color='IgM',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False,layer='dsb')
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='Celltype_L4_L5_Refine_R2',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='KLRB1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='IFNG',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='CXCR3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='CCR6',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='CD45RA',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCL5',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='CCR5',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='KLRB1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='CXCR5',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='CCR6',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.umap(adata, color='CD27',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',size=1)

In [ ]:
sc.pl.violin(adata, 'CD27',groupby=groupby,size=1)

In [ ]:
#Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LTK','PTPN13','PDE4D','CCR6','RORC','NR1D1','CTSH','KIF5C','LGALS3','USP10','CMTM6','TOB1',
                     'TNFSF13B','CISH','AQP3','AUTS2','NSG1','S100A4'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1/Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'GZMH','IL18RAP','S1PR5','LYAR','NKG7','CST7','PRF1','TBX21','LINC01871','KLRG1','MYBL1','EOMES',
                     'EFHD2','DUSP2','SAMD3','CTSW','ID2','MATK','HOPX',],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','TBX21','IFNG',
                      'CMC1','CST7','FCRL3','CCL4','SLAMF7','EOMES','PDCD1','NKG7','CCR5','KLRK1','F2R','PLEK'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th22
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','AHR',
                      'CRIP1','LGALS1','LGALS1','S100A10','S100A4','PI16','LMNA','ANXA5','ANXA2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th2
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'PTGDR2','SNED1','NEFL','GATA3','FXYD7','C1orf162','GDPD5','IL4R','CAPG',
                     'LGALS1','TNFSF10','TNFRSF4','PPP1R9B','CSGALNACT1','NIBAN1','ERN1','SORL1','RUNX2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','LAG3','HAVCR2','BTBD9',
                      'CCL5','CTLA4','CD40LG','TOX','ZNF683'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th from BD
#TNFSF8=CD30L B3GAT1=CD57  BTLA= CD272  SLAMF5=CD84 HAVCR2=CD365
sc.pl.dotplot(adata, ['CXCR5','IL6R','TNFSF8','NRP1','IL21R','B3GAT1','BCL6','MAF','STAT3','ICOS','PDCD1','TIGIT','BTLA','CD200','SLAMF1','CD84',#Tfh
                      'IL4','IL17F','IL17A','IL21',#tfh分泌
                      'GATA3','SMAD1','STAT6','SPI1','IRF4',#Th9
                      'IL9','IL10','CCL17','CCL22','TGFB1',#th9分泌
                      'HAVCR2','CXCR4','CCR3','CCR4','CCR8','PTGDR2','GATA3','STAT5A','STAT6','MAF','GFI1','IRF4','NOTCH1','NOTCH2','IL1RL1','IL17RB','IFNGR1','IFNGR2','TNFRSF8',#Th2
                      'IL2','IL5','IL6','IL10','IL13','IL31',#Th2分泌
                      'CCR4','CCR6','CCR10','AHR','PDGFRA','PDGFRB',#Th22
                      'IL22','TNF',#Th22分泌
                      'CXCR3','CCR5','KLRD1','TBX21','STAT1','STAT4','EOMES','RUNX3','FASLG','IL12RB1','IL12RB2','IL18R1','IL27RA','NOTCH3','TNFSF11','ICOS','HAVCR2','DPP4',#Th1
                      'LTB','LTA','PRF1','GZMB','GZMA','TNF','IFNG',#Th1分泌
                      'CCR4','CCR6','KLRB1','ICOS','HAVCR2','RORC','RORA','STAT3','RUNX1','BATF','IRF4','MAF','IL6R','IL13RA1','IL21R','IL23R',#Th17
                      'TNF','CCL20','IL17A','IL17F','IL21','IL22','IL24','IL26',#Th17分泌
                      ],
              standard_scale='obs',groupby=groupby)

In [ ]:
adata.obs['cluster_dummy']="NOT"
adata.obs.loc[adata.obs[groupby]=="6",'cluster_dummy'] = "YES"
sc.pl.umap(adata, color='cluster_dummy')

In [ ]:
adata.obs[groupby].value_counts()

In [ ]:
#Naïve CD4+ T 需要CD45RA+ CCR7=CD197hi CD62Lhi
# effector memory T cells(TEM, CD45RA-/CCR7-)
# TEMRA cells, which are T cells that re-express CD45RA(CD45RA+/CCR7-)

#'Th22':#CCR4+ CCR6+ CCR10+ KLRB1- GZMK- GATA3lo CCL5- GATA3+,CCR4+ CCR6+ KLRB1- GZMKlo AHRlo
#'Th1/Th17':# CCR6+ CCR4- CXCR3+ KLRB1+ GZMK+ CCL5+
#'Th17':#step1 CCR6+ CCR4- KLRB1+ GZMK- CCL5- RORC+ GZMK- TBX21-
#'Th2':#step1 CCR4+ CCR6- KLRB1- GATA3+ GZMK- CCL5-；CCR4+ CXCR5- GATA3+ CCL5-.CD45RA- CD279- TBX21- LEF1+ ,且没有CCL5- GZMH- GZMK- GZMB- KLRB1-各种标记的情况下CD62L+ CD27+ CD25+ 
#'Th1':#step3 确认 CXCR3+ CCR6- KLRB1- GZMK+ CD279+ GZMK+ CCL5+. CD279+(核心) GZMK+(核心) TBX21+ CCL5+(核心) TIGIT+ KLRB1-(核心) CD127-/IL7Rlo CCR7-CD197- SELLlo/CD62Llo  GZMH- GNLY- PRF1- GZMB- 的是Th1

cell_dict = {#'Th22':['',],#CCR4+ CCR6+ CCR10+ KLRB1lo/- GZMK-    CRIP1 LGALS1 LGALS3 PI16 ANXA5 ANXA2
             #'Th2|Th22':[''],#PTGDR2=CRTH2 GATA3++ IL4R+ CCR4+ CCR6- KLRB1- GZMK- CCL5- CD62L+ CD25+     PTGDR2,SNED1,NEFL,GATA3,FXYD7,C1orf162
             #'Th1':['',],#EOMES+ GZMK+ CCL5+ KLRB1- CXCR3+ CCR6- CD279+ TBX21+ IFNG+.  CMC1,CST7,FCRL3,CCL4,SLAMF7,EOMES,PDCD1,NKG7,CCR5,KLRK1,F2R,PLEK
             #'Th17':[],#CCR6=CD196++ RORC+ CCR4- KLRB1+ GZMK- CCL5- ICOS+    高表达LTK,PTPN13,PDE4D,CCR6,RORC,NR1D1,CTSH,KIF5C,LGALS3,USP10,CMTM6,TOB1
             'Th1/Th17':['0','3','4','5','9','10',# CCL5+ GZMK+ KLRB1+
                         '7',#ISG15 CCL5+ GZMK+ KLRB1+
                         
                         '13',#CCL5+ GZMK+ KLRB1- CCR6+; Th17 不表达GZMK
                         '14',#CCL5+ GZMK- KLRB1+ CCR6lo; Th1 不表达KLRB1
                         '11',#CCL5+  GZMK-  KLRB1- CD279- CCR6- CCR7- ;CCL5局限到Th1/TH17.1, Th1表达PDCD1
                         '2',#CCL5+ GZMK+ KLRB1- CD279-;CCL5局限到Th1/TH17.1, Th1表达PDCD1, Th17不表达GZMK
                         #CCL5- GZMK- KLRB1+ CCR6- CD279+ 
                        
                        ],#DPP4+ CCR6+ EOMESlo CCR6lo CCR4- CXCR3+ KLRB1+ GZMKlo/+ CCL5+ TBX21+ 与Th1相似, 差异基因中等表达
    
    'CD27+ Th1':['8'],#确认 CCL5+  GZMK+  KLRB1- CD279+ TIGIT+ CCR6- CCR7lo TOX ; Tfh不表达TOX; 

    'CD27- Th17':['6'],#CCL5- GZMK- KLRB1+ CCR6+; KLRB1局限到Th17/Th17.1, EOMES-是Th17

    'Tfh':['12',#CCL5-  GZMK-  KLRB1- CD279- CCR6- CCR7lo
          '15'],#CCL5- GZMK- KLRB1-  CD279+ CCR6- CCR7lo CD272=BTLA TOX
    'Doublet|Lowquality':['17','18','19',#Neu lncRNA
                          '16',#lncRNA
                          '1'],#ADT
            }

In [ ]:
check_dict_duplicates(cell_dict)

In [ ]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R5'] = i

In [ ]:
(adata.obs['Celltype_L4_L5_Refine_R5'].isna()).value_counts()

In [ ]:
adata.obs['Celltype_L4_L5_Refine_R5'].value_counts()

In [ ]:
adata = adata[adata.obs['Celltype_L4_L5_Refine_R5'] != "Doublet|Lowquality",:]

In [ ]:
adata.obs['Celltype_L4_L5_Refine_R5'].value_counts()

In [ ]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine',
                           'Celltype_L3_L4_Refine','Celltype_L4_L5_Refine',
                           'Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3',
                           'Celltype_L4_L5_Refine_R4','Celltype_L4_L5_Refine_R5',
                           groupby,'receptor_type','receptor_type_BCR']]
indices['UMAP_1'] = adata.obsm['X_umap'][:, 0].copy()  #
indices['UMAP_2'] = adata.obsm['X_umap'][:, 1].copy()  #
indices.rename(columns={groupby: 'leiden_cluster'}, inplace=True) #Save the cluster categorical, check the relationship between clusters and clinical to avoid missing someone.
indices['leiden_cluster'] = celltype + " c" + indices['leiden_cluster'].astype(str)
os.makedirs(f'{finnal_path}/{celltype}', exist_ok=True)
indices.to_csv(f"{finnal_path}/{celltype}/Finnal_indices_{celltype}.csv")
indices.head()

# 6. 定 终 ProliferativeT Cell Refine

In [67]:
celltype="ProliferativeT"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [68]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3','Celltype_L4_L5_Refine_R4',],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4,
           legend_loc='on data')

In [69]:
groupby = "L4_leiden_TOTALVI_1.5"

In [420]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='S.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [422]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='obs',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='Phase',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CTLA4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='FOXP3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='CTLA4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='PDCD1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='TIGIT',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='IL2RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
adata.obs.groupby(['Phase',groupby]).size()

In [ ]:
sc.pl.umap(adata, color='Phase',legend_fontsize=4, legend_fontoutline=2,size=2)

In [ ]:
#s.genes
sc.pl.dotplot(adata, ["MCM5","PCNA","TYMS","FEN1","MCM7","MCM4","RRM1","UNG","GINS2","MCM6","CDCA7",
"DTL","PRIM1","UHRF1","CENPU","HELLS","RFC2","POLR1B","NASP","RAD51AP1","GMNN","WDR76",
"SLBP","CCNE2","UBR7","POLD3","MSH2","ATAD2","RAD51","RRM2","CDC45","CDC6","EXO1",
"TIPIN","DSCC1","BLM","CASP8AP2","USP1","CLSPN","POLA1","CHAF1B","MRPL36","E2F8",],
              standard_scale='obs',groupby=groupby)

In [ ]:
#g2m.genes
sc.pl.dotplot(adata, ["HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80","CKS2","NUF2","CKS1B","MKI67",
"TMPO","CENPF","TACC3","PIMREG","SMC4","CCNB2","CKAP2L","CKAP2","AURKB","BUB1","KIF11","ANP32E",
"TUBB4B","GTSE1","KIF20B","HJURP","CDCA3","JPT1","CDC20","TTK","CDC25C","KIF2C","RANGAP1","NCAPD2",
"DLGAP5","CDCA2","CDCA8","ECT2","KIF23","HMMR","AURKA","PSRC1","ANLN","LBR","CKAP5","CENPE",
"CTCF","NEK2","G2E3","GAS2L3","CBX5","CENPA",],
              standard_scale='obs',groupby=groupby)

In [ ]:
sc.pl.dotplot(adata, ['MKI67','STMN1','PCNA','CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMB','BTBD9','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','CCR4',
                      'CCL5','CTLA4','CD40LG'],
              standard_scale='obs',groupby=groupby)

In [ ]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

In [70]:
#增殖CD38 MKI67 STMN1

cell_dict = {'Proliferative CD4+ Treg':['1','2','12'],
            'Proliferative CD8+ memory T cells':['11',#GZMK Tem
                                         '17',#GZMB Tem
                                         '14',#GZMK-GZMB- Tcm
                                         '9',#GZMB Temra
                                         '16',#Temra
                                          ],
            'Proliferative help memory T cells':['3','4','5','6','7',
                                         '0','10',#CCL5+ GZMK+ Th1
                                         '13',],
            'Proliferative DN T cells':['15'],#
            'Proliferative Vδ2+ T cells':['18'],#
             'Doublet|Lowquality':['8'],
            }

In [71]:
check_dict_duplicates(cell_dict)

'无重复'

In [72]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R5'] = i

In [73]:
(adata.obs['Celltype_L4_L5_Refine_R5'].isna()).value_counts()

Celltype_L4_L5_Refine_R5
False    48506
Name: count, dtype: int64

In [74]:
adata.obs.loc[adata.obs['Celltype_L4_L5_Refine']=='Proliferative Cytotoxic CD4+ T','Celltype_L4_L5_Refine_R5'] = 'Proliferative cytotoxic CD4+ T cells'

In [75]:
adata = adata[adata.obs['Celltype_L4_L5_Refine_R5'] != "Doublet|Lowquality",:]

In [76]:
adata.obs['Celltype_L4_L5_Refine_R5'].value_counts()

Celltype_L4_L5_Refine_R5
Proliferative help memory T cells       23240
Proliferative CD8+ memory T cells       13607
Proliferative CD4+ Treg                  9026
Proliferative DN T cells                  755
Proliferative cytotoxic CD4+ T cells      313
Proliferative Vδ2+ T cells                203
Name: count, dtype: int64

In [77]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine',
                           'Celltype_L3_L4_Refine','Celltype_L4_L5_Refine',
                           'Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3',
                           'Celltype_L4_L5_Refine_R4','Celltype_L4_L5_Refine_R5',
                           groupby,'receptor_type','receptor_type_BCR']]
indices['UMAP_1'] = adata.obsm['X_umap'][:, 0].copy()  #
indices['UMAP_2'] = adata.obsm['X_umap'][:, 1].copy()  #
indices.rename(columns={groupby: 'leiden_cluster'}, inplace=True) #Save the cluster categorical, check the relationship between clusters and clinical to avoid missing someone.
indices['leiden_cluster'] = celltype + " c" + indices['leiden_cluster'].astype(str)
os.makedirs(f'{finnal_path}/{celltype}', exist_ok=True)
indices.to_csv(f"{finnal_path}/{celltype}/Finnal_indices_{celltype}.csv")
indices.head()

# No Run